# Manual security review worksheet

**MSR 2027 · SkillGuard · RQ1 — manual validation of Cisco Skill Scanner findings**

This notebook turns the distinct security finding locations of one completed Cisco run into a **review worksheet**
that human reviewers fill in. For every location it shows the exact code or instructions (taken from the exported
skill files, with line numbers), what Cisco detected and why, and the trust boundary involved.

It keeps three kinds of information strictly apart:

| kind | where it comes from | columns |
|---|---|---|
| **Cisco output** | the saved run: `individual_findings.csv`, `manual_review_template.csv`, raw JSON | `rule_id`, `severity`, `cisco_*`, `reported_*` |
| **Suggested interpretation** | an analyst draft in `inputs/suggested_interpretations.json` (AI-assisted, not human-reviewed) | `suggested_*` |
| **Human-reviewed labels** | filled in by reviewers in the worksheet | `reviewer_1_*`, `reviewer_2_*`, `final_*` (blank here) |

Nothing in this notebook labels a finding as a true positive, false positive or false negative.

**Guarantees.** The notebook only reads the saved run, the suggestions file and the exported skill files. It never runs
Cisco, never executes a skill script, never touches the GitSkills database and never modifies the source run. Every
quoted file is checked against the SHA-256 recorded in the skill's `manifest.json` before it is quoted. An existing
worksheet is never overwritten, because reviewers may already have written labels into it.

## 1 — Settings

* `SOURCE_RUN` — the completed Cisco run folder (relative to the repository root).
* `SUGGESTIONS` — the analyst draft with one entry per review ID (priority, excerpts to quote, trust boundary,
  suggested interpretation).
* `OUTPUT_DIR` — where the worksheet is written. If a worksheet already exists there, the notebook stops.

In [ ]:
SOURCE_RUN = "results/cisco_dataset_runs/agent_skills_sample/runs/2026-11-03_173516_sample30-seed2027"
SUGGESTIONS = "results/manual_review/sample30-seed2027/inputs/suggested_interpretations.json"
OUTPUT_DIR = "results/manual_review/sample30-seed2027"

## 2 — Load the run and the suggestions

The review IDs come from the run's `manual_review_template.csv` (one row per distinct security location seen in a
complete scan). The suggestions file must cover exactly those IDs, so no location can be silently dropped or added.

In [ ]:
import hashlib
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next(f for f in (cwd, *cwd.parents) if (f / "notebooks").is_dir() and (f / "results").is_dir())
RUN_DIR = ROOT / SOURCE_RUN
OUT_DIR = ROOT / OUTPUT_DIR


def show(path):
    return str(path.relative_to(ROOT)) if path.is_relative_to(ROOT) else str(path)


def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


template = pd.read_csv(RUN_DIR / "manual_review_template.csv")
findings = pd.read_csv(RUN_DIR / "individual_findings.csv")
scans = pd.read_csv(RUN_DIR / "scan_summary.csv")
run_metadata = json.loads((RUN_DIR / "run_metadata.json").read_text(encoding="utf-8"))
suggestions_doc = json.loads((ROOT / SUGGESTIONS).read_text(encoding="utf-8"))
suggestions = suggestions_doc["findings"]

missing = sorted(set(template["review_id"]) - set(suggestions))
extra = sorted(set(suggestions) - set(template["review_id"]))
if missing or extra:
    raise ValueError(f"Suggestions do not match the review template. Missing: {missing}. Not in template: {extra}.")
if template["only_in_non_complete_scans"].astype(str).str.lower().eq("true").any():
    print("Note: some locations were seen only in contract-only or partial scans.")

print(f"Source run    : {SOURCE_RUN}")
print(f"Scanner       : {sorted(scans['scanner_version'].dropna().unique())}, policy fingerprint "
      f"{sorted(scans['policy_fingerprint'].dropna().str[:12].unique())}")
print(f"Locations     : {len(template)} distinct security locations in {template['skill_id'].nunique()} skill(s)")
print(f"Suggestions   : {suggestions_doc['about']['suggested_by']}")

## 3 — Check the quoted files and extract excerpts

For each excerpt the notebook finds the file in the run's `full/` folder (the full-folder export, which also contains
`SKILL.md`), checks its SHA-256 against the skill's `manifest.json`, and quotes the requested lines with their line
numbers. Lines marked `>>` are where the evidence actually is (`verified_evidence_line`); Cisco's own reported line is
kept in `reported_line_number` and is never replaced. Secret-looking values are redacted with the same patterns as
notebook 01 (none were present in this run's excerpts).

In [ ]:
SECRET_PATTERNS = [
    (re.compile(r"(?i)((?:password|passwd|secret|token|api[_-]?key|access[_-]?key)\w*['\"]?\s*[=:]\s*)(['\"])([^'\"\s]{4,})\2"), r"\1\2[REDACTED]\2"),
    (re.compile(r"(?i)((?:password|passwd|secret|token|api[_-]?key)\w*\s*=\s*)(?![$%{(])([A-Za-z0-9_\-/+]{8,})(?![\w.(\[])"), r"\1[REDACTED]"),
    (re.compile(r"(://[^/\s:@]+:)[^@\s]+@"), r"\1[REDACTED]@"),
    (re.compile(r"\b(AKIA|ASIA|ghp_|gho_|github_pat_|xox[bap]-|AIza|sk-)[A-Za-z0-9_\-]{8,}"), r"\1[REDACTED]"),
]


def redact(text):
    if not isinstance(text, str):
        return text
    for pattern, replacement in SECRET_PATTERNS:
        text = pattern.sub(replacement, text)
    return text


manifests = {sid: json.loads((RUN_DIR / sid / "manifest.json").read_text(encoding="utf-8")) for sid in template["skill_id"].unique()}
file_checks = []


def checked_file(skill_id, rel_path):
    """Path of an exported file after checking its SHA-256 against the skill's manifest.json."""
    recorded = {f["path"]: f["sha256"] for f in manifests[skill_id]["files"]}
    path = RUN_DIR / skill_id / "full" / rel_path
    if rel_path not in recorded:
        raise ValueError(f"{skill_id}: {rel_path} is not in manifest.json")
    actual = sha256_of(path)
    if actual != recorded[rel_path]:
        raise ValueError(f"{skill_id}: {rel_path} does not match its recorded SHA-256")
    file_checks.append({"skill_id": skill_id, "file": rel_path, "sha256": actual})
    return path


def excerpt(skill_id, spec, mark_lines):
    lines = checked_file(skill_id, spec["file"]).read_text(encoding="utf-8").splitlines()
    if not 1 <= spec["start"] <= spec["end"] <= len(lines):
        raise ValueError(f"{skill_id}: lines {spec['start']}-{spec['end']} outside {spec['file']} ({len(lines)} lines)")
    body = "\n".join(f"{'>>' if n in mark_lines and spec['file'] == mark_lines[n] else '  '} {n:>4} | {lines[n - 1]}"
                     for n in range(spec["start"], spec["end"] + 1))
    return {"file": spec["file"], "start": spec["start"], "end": spec["end"], "why": spec["why"], "text": redact(body)}


excerpts = {}
for row in template.itertuples():
    s = suggestions[row.review_id]
    mark = {int(s["verified_evidence_line"]): row.file_path} if s.get("verified_evidence_line") else {}
    excerpts[row.review_id] = [excerpt(row.skill_id, spec, mark) for spec in s["excerpts"]]

file_checks_df = pd.DataFrame(file_checks).drop_duplicates()
print(f"{len(file_checks_df)} distinct file(s) quoted; all match the SHA-256 in their manifest.json")

## 4 — Build the worksheet

One row per distinct security location. Column groups, from left to right:

1. **Identification and priority** — `review_id`, review `priority` (1 = inspect first) and the `behavior_group`
   that joins locations describing the same underlying behaviour.
2. **Cisco output** — rule, severity, category, analyzer, the configurations that reported it, reported file and line,
   Cisco's snippet, description and matched pattern, and the automatic evidence check from notebook 01.
3. **Excerpts** — the quoted code or instructions with line numbers.
4. **Suggested interpretation (draft)** — what Cisco detected in plain words, the mechanism behind the rule, the
   trust boundary, the data flow, a suggested label with confidence and rationale, an alternative reading, open
   questions, and behaviour seen nearby that Cisco did not report (an observation for the coverage study, not a
   false-negative claim).
5. **Human review (blank)** — two independent reviewers, then the adjudicated final label.

In [ ]:
LABELS = list(suggestions_doc["about"]["label_vocabulary"])
EVIDENCE_VALUES = list(suggestions_doc["about"]["evidence_present_vocabulary"])
HUMAN_COLUMNS = ["reviewer_1", "reviewer_1_evidence_present", "reviewer_1_label", "reviewer_1_rationale",
                 "reviewer_2", "reviewer_2_evidence_present", "reviewer_2_label", "reviewer_2_rationale",
                 "final_evidence_present", "final_label", "final_rationale", "adjudicated_by", "review_date"]

by_finding_id = findings.drop_duplicates("finding_id").set_index("finding_id")
rows = []
for row in template.itertuples():
    s = suggestions[row.review_id]
    f = by_finding_id.loc[str(row.cisco_finding_ids).split(";")[0].strip()]
    if s["suggested_interpretation"] not in LABELS:
        raise ValueError(f"{row.review_id}: unknown suggested label {s['suggested_interpretation']!r}")
    rows.append({
        "review_id": row.review_id, "priority": s["priority"], "behavior_group": s["behavior_group"],
        "skill_id": row.skill_id, "repo": row.repo, "skill_path": row.skill_path,
        "sample_stratum": manifests[row.skill_id].get("sample_stratum"),
        "rule_id": row.rule_id, "severity_reported": row.severity_reported, "cisco_category": row.cisco_category,
        "analyzer": row.analyzer, "configurations": row.configurations, "run_statuses": row.run_statuses,
        "file_path": row.file_path, "reported_line_number": row.reported_line_number,
        "verified_evidence_line": s.get("verified_evidence_line"),
        "evidence_status_auto": row.evidence_status,
        "cisco_snippet": redact(f["snippet"]) if pd.notna(f["snippet"]) else None,   # already redacted by notebook 01
        "cisco_snippet_was_redacted": bool(f["snippet_redacted"]),
        "cisco_description": redact(f["description"]), "cisco_matched_pattern": f["meta_matched_pattern"],
        "excerpts": "\n\n".join(f"# {e['file']} lines {e['start']}-{e['end']}: {e['why']}\n{e['text']}" for e in excerpts[row.review_id]),
        "what_cisco_detected": s["what_cisco_detected"], "cisco_mechanism": s["cisco_mechanism"],
        "trust_boundary": s["trust_boundary"], "data_flow": s["data_flow"],
        "suggested_interpretation": s["suggested_interpretation"], "suggested_confidence": s["suggested_confidence"],
        "suggested_rationale": s["suggested_rationale"], "alternative_interpretation": s.get("alternative_interpretation") or None,
        "open_questions": s.get("open_questions") or None,
        "related_review_ids": "; ".join(s.get("related_review_ids") or []) or None,
        "observed_not_reported": s.get("observed_not_reported") or None,
        "suggestion_source": suggestions_doc["about"]["suggested_by"],
        **dict.fromkeys(HUMAN_COLUMNS),
    })

worksheet = pd.DataFrame(rows).sort_values("priority").reset_index(drop=True)
if sorted(worksheet["priority"]) != list(range(1, len(worksheet) + 1)):
    raise ValueError("priorities must be 1..n without gaps or ties")

display(worksheet[["priority", "review_id", "skill_id", "rule_id", "severity_reported", "file_path",
                   "reported_line_number", "verified_evidence_line", "suggested_interpretation", "suggested_confidence"]])

## 5 — Behaviour-level view

Cisco reports one finding per rule and line, so a single behaviour can appear several times (two lines of the same
HTTP request, or the same code block reported by a static and a behavioral rule). The behaviour groups below are the
analyst's draft grouping; reviewers should confirm or split them. Counts here are **suggestions, not results**.

In [ ]:
behaviors = (worksheet.groupby("behavior_group", sort=False)
             .agg(skill=("skill_id", "first"), locations=("review_id", "size"),
                  rules=("rule_id", lambda s: ", ".join(dict.fromkeys(s))),
                  max_severity=("severity_reported", lambda s: max(s, key=["INFO", "LOW", "MEDIUM", "HIGH", "CRITICAL"].index)),
                  suggested=("suggested_interpretation", lambda s: ", ".join(sorted(set(s))))))
display(behaviors)
print(f"{len(worksheet)} locations -> {len(behaviors)} draft behaviour groups in {worksheet['skill_id'].nunique()} skills")
print("\nSuggested interpretations (draft, not human-reviewed):")
display(worksheet["suggested_interpretation"].value_counts().reindex(LABELS, fill_value=0).rename("locations").to_frame())

## 6 — Write the worksheet

Files written to `OUTPUT_DIR`:

| file | content |
|---|---|
| `manual_review_worksheet.csv` | the worksheet; reviewers fill the blank `reviewer_*` and `final_*` columns |
| `manual_review_worksheet.md` | the same content for reading: codebook, priority list, one section per location with code |
| `worksheet_metadata.json` | source run, SHA-256 of every input and quoted file, generation time |

If `manual_review_worksheet.csv` already exists the notebook stops instead of overwriting reviewer labels. To
regenerate, move the old worksheet away first (after copying any labels you want to keep).

In [ ]:
csv_path, md_path, meta_path = OUT_DIR / "manual_review_worksheet.csv", OUT_DIR / "manual_review_worksheet.md", OUT_DIR / "worksheet_metadata.json"
existing = [p for p in (csv_path, md_path, meta_path) if p.exists()]
if existing:
    raise FileExistsError("Worksheet files already exist and may contain reviewer labels; nothing was written: "
                          + ", ".join(show(p) for p in existing))


def md_escape(value):
    return str(value).replace("|", "\\|").replace("\n", " ") if value is not None and not (isinstance(value, float) and pd.isna(value)) else ""


def line_text(value):
    return "n/a" if value is None or (isinstance(value, float) and pd.isna(value)) else str(int(value))


about = suggestions_doc["about"]
M = [f"# Manual security review worksheet: `{RUN_DIR.name}`\n",
     f"Generated by `notebooks/04_manual_review_worksheet.ipynb` on {datetime.now(timezone.utc).isoformat(timespec='seconds')} "
     f"from the saved Cisco run `{SOURCE_RUN}` (scanner {', '.join(sorted(scans['scanner_version'].dropna().unique()))}, "
     f"policy fingerprint {', '.join(sorted(scans['policy_fingerprint'].dropna().str[:12].unique()))}). "
     f"It covers {len(worksheet)} distinct security finding locations from complete scans in "
     f"{worksheet['skill_id'].nunique()} skills. Cisco findings are rule matches, not confirmed vulnerabilities.\n",
     "**How to read this file.** Each section separates three things: *Cisco output* (what the scanner reported), "
     "the *suggested interpretation* (an AI-assisted analyst draft, not human-reviewed), and the *human review* "
     "fields, which are blank and are filled in `manual_review_worksheet.csv`. No finding is labelled a true positive, "
     "false positive or false negative here.\n",
     "## Codebook\n",
     f"**Threat model.** {about['threat_model']}\n",
     "**Evidence present** (does the primitive the rule names exist at or next to the reported location?)\n",
     "| value | meaning |\n|---|---|", *[f"| `{k}` | {v} |" for k, v in about["evidence_present_vocabulary"].items()],
     "\n**Interpretation label**\n",
     "| label | meaning |\n|---|---|", *[f"| `{k}` | {v} |" for k, v in about["label_vocabulary"].items()],
     "\n**Procedure.** Two reviewers label every row independently (without seeing each other's labels), recording "
     "`reviewer_N_evidence_present`, `reviewer_N_label` and a one-sentence rationale. Disagreements are resolved by "
     "discussion and recorded in `final_*` with `adjudicated_by`. Agreement (Cohen's kappa) is computed on the "
     "independent labels before adjudication.\n",
     "## Review order\n",
     "| priority | review_id | skill | rule | severity | location | suggested (draft) | why this priority |",
     "|---|---|---|---|---|---|---|---|"]
for r in worksheet.itertuples():
    loc = f"{r.file_path}:{line_text(r.reported_line_number)}"
    M.append(f"| {r.priority} | `{r.review_id}` | {md_escape(r.skill_id)} | {r.rule_id} | {r.severity_reported} | {md_escape(loc)} | "
             f"{r.suggested_interpretation} ({r.suggested_confidence}) | {md_escape(suggestions[r.review_id]['priority_reason'])} |")
M += ["\n## Behaviour groups (draft)\n",
      "| behaviour group | locations | rules | max severity | suggested (draft) |", "|---|---|---|---|---|"]
M += [f"| {md_escape(g)} | {b.locations} | {md_escape(b.rules)} | {b.max_severity} | {md_escape(b.suggested)} |" for g, b in behaviors.iterrows()]

for r in worksheet.itertuples():
    s = suggestions[r.review_id]
    M += [f"\n---\n\n## {r.priority}. `{r.review_id}` — {r.rule_id} ({r.severity_reported}) in {r.skill_id}\n",
          f"**Skill:** `{r.repo}` / `{r.skill_path}` (stratum: {r.sample_stratum}). **Behaviour group:** {r.behavior_group}.\n",
          "### Cisco output\n",
          "| field | value |\n|---|---|",
          f"| rule / category / analyzer | `{r.rule_id}` / {r.cisco_category} / {r.analyzer} |",
          f"| reported in configurations | {md_escape(r.configurations)} |",
          f"| reported file and line | `{r.file_path}` line {line_text(r.reported_line_number)} |",
          f"| evidence check (notebook 01) | {r.evidence_status_auto}; evidence found on line {line_text(r.verified_evidence_line)} |",
          f"| Cisco snippet | `{md_escape(r.cisco_snippet)}` |" if isinstance(r.cisco_snippet, str) else "| Cisco snippet | none (Cisco reported no line or snippet) |",
          f"| Cisco description | {md_escape(r.cisco_description)} |",
          *([f"| matched pattern | `{md_escape(r.cisco_matched_pattern)}` |"] if isinstance(r.cisco_matched_pattern, str) else []),
          "\n### Code and instructions\n"]
    for e in excerpts[r.review_id]:
        M += [f"`{e['file']}` lines {e['start']}-{e['end']}: {e['why']}\n", "```text", e["text"], "```\n"]
    M += ["### Suggested interpretation (draft, not human-reviewed)\n",
          f"- **What Cisco detected:** {s['what_cisco_detected']}",
          f"- **Rule mechanism:** {s['cisco_mechanism']}",
          f"- **Trust boundary:** {s['trust_boundary']}",
          f"- **Data flow:** {s['data_flow']}",
          f"- **Suggested label:** `{s['suggested_interpretation']}` (confidence: {s['suggested_confidence']})",
          f"- **Rationale:** {s['suggested_rationale']}",
          *([f"- **Alternative reading:** {s['alternative_interpretation']}"] if s.get("alternative_interpretation") else []),
          *([f"- **Questions for the reviewer:** {s['open_questions']}"] if s.get("open_questions") else []),
          *([f"- **Related locations:** {', '.join('`' + x + '`' for x in s['related_review_ids'])}"] if s.get("related_review_ids") else []),
          *([f"- **Seen nearby but not reported by Cisco (coverage observation, not a false-negative claim):** {s['observed_not_reported']}"]
            if s.get("observed_not_reported") else []),
          "\n### Human review\n",
          "Fill in `manual_review_worksheet.csv`: `reviewer_1_*`, `reviewer_2_*`, then `final_*`.\n"]

OUT_DIR.mkdir(parents=True, exist_ok=True)
worksheet.to_csv(csv_path, index=False)
md_path.write_text("\n".join(M) + "\n", encoding="utf-8")
metadata = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "generated_by": "notebooks/04_manual_review_worksheet.ipynb",
    "source_run": SOURCE_RUN,
    "source_run_id": run_metadata.get("run_id"),
    "inputs": {name: {"path": show(path), "sha256": sha256_of(path)} for name, path in
               {"manual_review_template": RUN_DIR / "manual_review_template.csv", "individual_findings": RUN_DIR / "individual_findings.csv",
                "scan_summary": RUN_DIR / "scan_summary.csv", "suggestions": ROOT / SUGGESTIONS}.items()},
    "quoted_files": file_checks_df.to_dict("records"),
    "locations": len(worksheet), "behavior_groups": len(behaviors),
    "suggestion_source": about["suggested_by"],
    "human_columns": HUMAN_COLUMNS, "label_vocabulary": LABELS, "evidence_present_vocabulary": EVIDENCE_VALUES,
}
meta_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
for p in (csv_path, md_path, meta_path):
    print("wrote", show(p))

## 7 — After the review

Once both reviewers have filled in the worksheet:

1. Compute agreement on `reviewer_1_label` vs `reviewer_2_label` and on the evidence-present columns (Cohen's kappa,
   with the number of rows) before adjudication.
2. Report final labels per **location** and per **behaviour group**, and keep the suggested column out of every
   result table: it documents the draft, not a finding.
3. Report rule-match precision only as "share of reviewed locations where the primitive was present" and
   "share judged scanner/parser issues"; it is not vulnerability precision, and nothing here measures recall.
4. Behaviours listed under "seen nearby but not reported" are input for a separate, blind coverage protocol. They
   are not false negatives.